# Ripple-Spindle Sequence Coupling: First Hour of Postsleep

Sequential ripple -> spindle coupling with lag histograms. The overlap version ("spindle with ripple") is in `Task2b_ripple_spindle_overlap_full.ipynb`.

Yixiao: make the histogram first, then decide if it's different enough from the overlap measure.

First hour of postsleep only, same steps as `Task2_ripple_spindle_sequence.ipynb`.

---
## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# lab repo root on the path, so `modules` imports work from this folder
REPO_ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "modules" / "project_config.py").exists()
)
sys.path.insert(0, str(REPO_ROOT))
from modules.project_config import get_path  # noqa: E402

# local paths come from config.yaml (see config.example.yaml)
ROOT = get_path("R1_8_root")  # holds Rat_HM_Ephys_TD_Analysis_R1_8 and _R9_16
assert get_path("R9_16_root") == ROOT, "these notebooks expect R1_8_root == R9_16_root"
WAVE_CACHE = get_path("event_cache_root")
SESSION_MAP_CSV = get_path("session_map_csv")
OUT = REPO_ROOT / "results" / "leticia_event_coupling"  # figures and caches
SPINDLE_METHOD = "wavelet_amp_1_ampcore_3"
PHASES = ["postsleep"]

BOUT_KEYS = ["rat", "date", "trial", "nrem_bout_index"]

# ripples are HPC only: HPC ripple vs PL / RSC / HPC spindle (HPC-HPC same channel)
RIPPLE_AREA = "HPC"
AREAS = ["PL", "RSC", "HPC"]  # spindle areas
PAIRS = [(RIPPLE_AREA, s) for s in AREAS]

WINDOW_MS = (2, 1000)  # Ripple -> Spindle, peak-to-peak (PROJECT_DATA_REFERENCE.md)

# 7 rats with trusted ripple detection
RATS = [1, 3, 7, 8, 11, 12, 13]

FIGDIR = OUT / "task2_ripple_spindle_sequence_first_hour_figures"
FIGDIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    """Save a figure. Takes the figure object explicitly -- plt.gcf() is
    unreliable after tight_layout() and returns a blank figure after show()."""
    fig.savefig(FIGDIR / f"{name}.png", dpi=200, bbox_inches="tight")
    print("saved:", name)


print("drive reachable:", ROOT.exists())
print("rats in analysis:", RATS)
print("pairs:", PAIRS, "| window:", WINDOW_MS)

## Load and restrict

In [ ]:
from modules.event_coupling.build_tables import add_condition, build_wave

ripples = add_condition(
    build_wave(ROOT, "ripple", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)
spindles = add_condition(
    build_wave(ROOT, "spindle", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)

rip = ripples[
    (ripples["rat"].isin(RATS))
    & (ripples["group"] != "other")
    & (ripples["area"] == RIPPLE_AREA)
]
spn = spindles[
    (spindles["rat"].isin(RATS))
    & (spindles["group"] != "other")
    & (spindles["area"].isin(AREAS))
]

print(f"{len(rip):,} ripples | {len(spn):,} spindles")
print("ripples by rat:")
print(rip.groupby("rat").size().to_string())

## Check: is the first hour over-represented?

NREM per hour from the ripple bout info, to check the elapsed-time correction.

In [ ]:
TRIAL_LEN = 1_800_000  # 30 min at 1000Hz, confirmed from real trial lengths


def add_elapsed_time(df, index_col):
    """True elapsed time since the start of postsleep, correcting for
    peak_index resetting to 0 at the start of EACH trial file."""
    min_trial = df.groupby(["rat", "date"])["trial"].transform("min")
    return (df["trial"] - min_trial) * TRIAL_LEN + df[index_col]


def split_bout_by_hour(start, end):
    """Minutes of NREM in each of hours 1-4 for one bout [start, end)."""
    HOUR = 60 * 60 * 1000
    out = np.zeros(4)
    for h in range(4):
        lo, hi = h * HOUR, (h + 1) * HOUR
        overlap = max(0, min(end, hi) - max(start, lo))
        out[h] = overlap / 1000 / 60
    return out


bouts_check = rip[
    ["rat", "date", "trial", "nrem_bout_start_index", "nrem_bout_end_index"]
].drop_duplicates()
bouts_check["elapsed_start"] = add_elapsed_time(bouts_check, "nrem_bout_start_index")
bouts_check["elapsed_end"] = add_elapsed_time(bouts_check, "nrem_bout_end_index")

hour_mins = np.array(
    [
        split_bout_by_hour(s, e)
        for s, e in zip(bouts_check["elapsed_start"], bouts_check["elapsed_end"])
    ]
)
bouts_check[["hour1", "hour2", "hour3", "hour4"]] = hour_mins
per_rat_hours = bouts_check.groupby("rat")[["hour1", "hour2", "hour3", "hour4"]].sum()

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(per_rat_hours))
colors = ["#2c5f8a", "#5b9bd5", "#a9c9e8", "#d6e4f0"]
for i, hour_col in enumerate(["hour1", "hour2", "hour3", "hour4"]):
    ax.bar(
        per_rat_hours.index.astype(str),
        per_rat_hours[hour_col],
        bottom=bottom,
        label=f"Hour {i + 1}" if i < 3 else "Last hour",
        color=colors[i],
    )
    bottom += per_rat_hours[hour_col].values
ax.set_xlabel("Rat")
ax.set_ylabel("NREM duration (minutes)")
ax.set_title("Postsleep NREM duration by rat and recording hour (7 rats, ripple bouts)")
ax.legend(title="Recording hour")
plt.tight_layout()
plt.show()

## Restrict to the first hour of postsleep

Elapsed time = trial position x 30 min + within-trial index. `rip` and `spn` filtered separately.

In [ ]:
ONE_HOUR_SAMPLES = 60 * 60 * 1000  # 1000 Hz -> 1 sample index = 1 ms


def filter_first_hour_fixed(df, index_col):
    """Correct version: filters on true elapsed time (trial position x 30
    min + within-trial index), not raw peak_index."""
    df = df.copy()
    df["_elapsed"] = add_elapsed_time(df, index_col)
    return df[df["_elapsed"] < ONE_HOUR_SAMPLES].drop(columns="_elapsed")


rip_full, spn_full = rip, spn
rip = filter_first_hour_fixed(rip_full, "ripple_peak_index")
spn = filter_first_hour_fixed(spn_full, "spindle_peak_index")

for name, full, hr in [("ripples", rip_full, rip), ("spindles", spn_full, spn)]:
    print(
        f"{name}: {len(full):,} total -> {len(hr):,} in true first hour "
        f"({100 * len(hr) / len(full):.1f}%)"
    )

## Lag histogram setup

In [ ]:
from modules.event_coupling import lag_histogram as lh

MAX_LAG_MS = 1600
EDGES = np.arange(-1600, 1620, 20)
BASELINE_MS = 1400  # tail = |lag| >= 1400, clear of the 2-1000 ms window
left = EDGES[:-1]

WINDOWS = [("R->S", WINDOW_MS[0], WINDOW_MS[1])]

### Check before using

Self-test with a planted lag.

In [ ]:
lh.self_test(planted_lag_ms=200)

## Full dataset

Per pair. Baseline from the tails (|lag| >= 1400 ms), shown as a line.

`lag = spindle_peak - ripple_peak`: positive = spindle after ripple (R->S window shaded).

In [ ]:
res_full = {}
for r_area, s_area in PAIRS:
    pair_label = f"{r_area} ripple - {s_area} spindle"
    lg = lh.collect_lags(
        rip[rip["area"] == r_area],
        spn[spn["area"] == s_area],
        "ripple",
        "spindle",
        max_lag_ms=MAX_LAG_MS,
    )
    res_full[(r_area, s_area)] = lh.summarise(
        lg, EDGES, baseline_ms=BASELINE_MS, windows=WINDOWS, label=pair_label
    )
    del lg

print(f"{'pair':<24}{'ratio':>9}")
for r_area, s_area in PAIRS:
    key = (r_area, s_area)
    pair_label = f"{r_area} ripple - {s_area} spindle"
    f_ratio = res_full[key]["windows"]["R->S"]["ratio"]
    print(f"{pair_label:<24}{f_ratio:>9.3f}")

In [ ]:
for (r_area, s_area), r in res_full.items():
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(left, r["counts"], color="tab:blue", lw=1.6)
    ax.axhline(r["baseline"], color="k", lw=1, ls=":", label="baseline")
    ax.axvspan(
        WINDOW_MS[0], WINDOW_MS[1], color="tab:blue", alpha=0.10, label="R->S window"
    )
    ax.axvline(0, color="k", lw=1)
    ax.set_xlabel("spindle peak - ripple peak (ms)   [positive = spindle after ripple]")
    ax.set_ylabel("count")
    ax.set_title(f"{r_area} ripple vs {s_area} spindle -- full dataset")
    ax.legend(fontsize=8)
    plt.tight_layout()
    save_fig(fig, f"02_histogram_full_{r_area}_{s_area}")
    plt.show()

## Zoom around lag = 0

Same histogram, +/-500 ms.

In [ ]:
ZOOM_MS = 500  # +/- range for the zoomed view

for (r_area, s_area), r in res_full.items():
    y = r["counts"]
    visible = (left >= -ZOOM_MS) & (left <= ZOOM_MS)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(left, y, color="tab:blue", lw=1.6)
    ax.axhline(r["baseline"], color="k", lw=1, ls=":", label="baseline")
    ax.axvspan(
        WINDOW_MS[0], WINDOW_MS[1], color="tab:blue", alpha=0.10, label="R->S window"
    )
    ax.axvline(0, color="k", lw=1)
    ax.set_xlim(-ZOOM_MS, ZOOM_MS)
    pad = 0.05 * max(y[visible].max() - y[visible].min(), 1e-6)
    ax.set_ylim(y[visible].min() - pad, y[visible].max() + pad)
    ax.set_xlabel("spindle peak - ripple peak (ms)   [positive = spindle after ripple]")
    ax.set_ylabel("count")
    ax.set_title(
        f"{r_area} ripple vs {s_area} spindle -- full dataset, ZOOMED \u00b1{ZOOM_MS}ms"
    )
    ax.legend(fontsize=8)
    plt.tight_layout()
    save_fig(fig, f"03_histogram_full_ZOOM_{r_area}_{s_area}")
    plt.show()

In [ ]:
for key, r in res_full.items():
    print(f"{key}: peak at {r['peak_ms']} ms")

## Find the trough and second peak after the lag-0 spike

Trough after the 0 ms spike and the next peak; positions from a 3-bin (60 ms) moving average, heights from the raw curve.

In [ ]:
from scipy.ndimage import uniform_filter1d


def find_trough_and_peak(left, y, search_from=0, search_to=800, smooth_bins=3):
    """After the lag=0 coincidence spike, find the first trough (local
    minimum) and the first peak (local maximum) that follows it, searching
    only in [search_from, search_to) ms."""
    mask = (left >= search_from) & (left < search_to)
    x = left[mask]
    y_raw = y[mask]
    y_smooth = uniform_filter1d(y_raw, size=smooth_bins)

    trough_idx = np.argmin(y_smooth)
    trough_ms = int(x[trough_idx])
    trough_val = float(y_raw[trough_idx])

    after = x > trough_ms
    if not after.any():
        return trough_ms, trough_val, None, None
    peak_idx = np.argmax(y_smooth[after])
    peak_ms = int(x[after][peak_idx])
    peak_val = float(y_raw[after][peak_idx])

    return trough_ms, trough_val, peak_ms, peak_val


print(f"{'pair':<22}{'trough_ms':>11}{'trough_val':>12}{'peak_ms':>10}{'peak_val':>10}")
for (r_area, s_area), r in res_full.items():
    label = f"{r_area}-{s_area}"
    y = r["counts"] / r["baseline"]
    trough_ms, trough_val, peak_ms, peak_val = find_trough_and_peak(left, y)
    peak_ms_str = f"{peak_ms}" if peak_ms is not None else "n/a"
    peak_val_str = f"{peak_val:.2f}" if peak_val is not None else "n/a"
    print(
        f"{label:<22}{trough_ms:>11}{trough_val:>12.2f}{peak_ms_str:>10}{peak_val_str:>10}"
    )

## End-of-ripple -> start-of-spindle (Yixiao's check)

Overlap or sequence? Uses `ripple_end_index` -> `spindle_start_index`.

`lag = spindle_start - ripple_end`: lag < 0 overlap, lag >= 0 sequential.

In [ ]:
res_end_start = {}
for r_area, s_area in PAIRS:
    pair_label = f"{r_area} ripple end - {s_area} spindle start"
    lg = lh.collect_lags_cols(
        rip[rip["area"] == r_area],
        spn[spn["area"] == s_area],
        "ripple_end_index",
        "spindle_start_index",
        max_lag_ms=MAX_LAG_MS,
    )
    res_end_start[(r_area, s_area)] = lh.summarise(
        lg,
        EDGES,
        baseline_ms=BASELINE_MS,
        windows=[
            ("overlap (lag<0)", -MAX_LAG_MS, 0),
            ("sequential (lag>=0)", 0, MAX_LAG_MS),
        ],
        label=pair_label,
    )
    del lg

for key, r in res_end_start.items():
    ov = r["windows"]["overlap (lag<0)"]["observed"]
    sq = r["windows"]["sequential (lag>=0)"]["observed"]
    print(
        f"{key}: {ov:,} pairs overlap (lag<0) | {sq:,} pairs sequential (lag>=0) "
        f"| {100 * sq / (ov + sq):.1f}% sequential"
    )

In [ ]:
for (r_area, s_area), r in res_end_start.items():
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(left, r["counts"], color="tab:purple", lw=1.6)
    ax.axhline(r["baseline"], color="k", lw=1, ls=":", label="baseline")
    ax.axvline(0, color="k", lw=1.2)
    ax.axvspan(
        EDGES[0], 0, color="tab:red", alpha=0.06, label="overlap (already counted)"
    )
    ax.axvspan(0, EDGES[-1], color="tab:green", alpha=0.06, label="sequential (new)")
    ax.set_xlabel(
        "spindle start - ripple end (ms)   [<0 = overlapping, >=0 = sequential]"
    )
    ax.set_ylabel("count")
    ax.set_title(f"{r_area} ripple end -> {s_area} spindle start -- full dataset")
    ax.legend(fontsize=8)
    plt.tight_layout()
    save_fig(fig, f"03_histogram_end_start_{r_area}_{s_area}")
    plt.show()

## 2-1000 ms window (Yixiao)

End-start histogram zoomed to 2-1000 ms.

In [ ]:
SEQ_WINDOW_MS = (2, 1000)  # Yixiao, Oct 2026: ripple_end -> spindle_start

for (r_area, s_area), r in res_end_start.items():
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(left, r["counts"], color="tab:purple", lw=1.6)
    ax.axhline(r["baseline"], color="k", lw=1, ls=":", label="baseline")
    ax.axvspan(
        SEQ_WINDOW_MS[0],
        SEQ_WINDOW_MS[1],
        color="tab:green",
        alpha=0.12,
        label="window to test (2-1000 ms)",
    )
    ax.axvline(0, color="k", lw=1)
    ax.set_xlabel("spindle start - ripple end (ms)")
    ax.set_ylabel("count")
    ax.set_title(
        f"{r_area} ripple end -> {s_area} spindle start -- first hour, {r_area}/{s_area}"
    )
    ax.legend(fontsize=8)
    plt.tight_layout()
    save_fig(fig, f"04_histogram_window_{r_area}_{s_area}")
    plt.show()

## Fixed-window count (observed, no shuffling yet)

Pairs with `spindle_start - ripple_end` in [2, 1000) ms.

In [ ]:
print(f"{'pair':<24}{'count':>9}")
seq_window_counts = {}
for r_area, s_area in PAIRS:
    pair_label = f"{r_area} ripple - {s_area} spindle"
    lg = lh.collect_lags_cols(
        rip[rip["area"] == r_area],
        spn[spn["area"] == s_area],
        "ripple_end_index",
        "spindle_start_index",
        max_lag_ms=MAX_LAG_MS,
    )
    n = int(np.sum((lg >= SEQ_WINDOW_MS[0]) & (lg < SEQ_WINDOW_MS[1])))
    seq_window_counts[(r_area, s_area)] = n
    print(f"{pair_label:<24}{n:>9,}")
    del lg

## Circular-shift surrogate for the fixed-window count

Spindle train shifted within each recording's NREM time, `N_ITER` times; z and one-sided p from `surrogate.window_significance`.

Own checkpoint folder (`task2_ripple_spindle_sequence_first_hour_checkpoints`).

In [ ]:
import pickle

from modules.event_coupling import surrogate as sg

N_ITER = 1000  # same as the full dataset

CKPT_DIR = OUT / "task2_ripple_spindle_sequence_first_hour_checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)


def make_checkpoint_fn(tag):
    path = CKPT_DIR / f"{tag}.pkl"

    def _fn(partial):
        with open(path, "wb") as f:
            pickle.dump(partial, f)
        print(f"    checkpoint saved: {path.name} ({partial['n_done']}/{N_ITER} iters)")

    return _fn


res_seq_surr = {}
for r_area, s_area in PAIRS:
    pair_label = f"{r_area} ripple - {s_area} spindle"
    print(f"--- {pair_label} ---")
    ckpt_path = CKPT_DIR / f"seqwindow_{r_area}_{s_area}.pkl"
    resume = None
    if ckpt_path.exists():
        with open(ckpt_path, "rb") as f:
            resume = pickle.load(f)
        print(f"    resuming from {resume['n_done']}/{N_ITER} iterations")

    if resume is not None and resume["n_done"] >= N_ITER:
        print(
            f"    already complete at {resume['n_done']}/{N_ITER} -- using checkpoint directly"
        )
        dist = resume
    else:
        dist = sg.surrogate_distribution_cols(
            rip[rip["area"] == r_area],
            spn[spn["area"] == s_area],
            "ripple_end_index",
            "spindle_start_index",
            EDGES,
            n_iter=N_ITER,
            seed=0,
            max_lag_ms=MAX_LAG_MS,
            scope="recording",
            verbose=True,
            checkpoint_every=50,
            checkpoint_fn=make_checkpoint_fn(f"seqwindow_{r_area}_{s_area}"),
            resume_from=resume,
        )

    res_seq_surr[(r_area, s_area)] = sg.window_significance(
        dist, EDGES, window_ms=SEQ_WINDOW_MS
    )

print(f"{'pair':<24}{'observed':>10}{'null mean':>11}{'p':>8}{'sig?':>7}")
for (r_area, s_area), r in res_seq_surr.items():
    pair_label = f"{r_area} ripple - {s_area} spindle"
    print(
        f"{pair_label:<24}{r['observed']:>10,}{r['null_mean']:>11,.1f}"
        f"{r['p']:>8.3f}{str(r['significant']):>7}"
    )

## HC vs Training

Per rat, paired (HC vs Training), 2-1000 ms end-start window. With 7 rats a sign test needs 7/7 for p=0.008.

Fewer events per rat in the first hour, so check the "rats with enough pairs per cell" table.

In [ ]:
# rat-days with < 10 min NREM excluded (Yixiao/Lisa, Oct 2026), pair counts shown per cell
MIN_NREM_MIN = 10  # minutes of NREM sleep required for a rat-day to count

# hour-1 NREM minutes from the unrestricted bout list (rip_full)
bouts_q = rip_full[
    ["rat", "date", "trial", "nrem_bout_start_index", "nrem_bout_end_index"]
].drop_duplicates()
bouts_q["elapsed_start"] = add_elapsed_time(bouts_q, "nrem_bout_start_index")
bouts_q["elapsed_end"] = add_elapsed_time(bouts_q, "nrem_bout_end_index")
bouts_q["hour1_min"] = [
    split_bout_by_hour(s, e)[0]
    for s, e in zip(bouts_q["elapsed_start"], bouts_q["elapsed_end"])
]
day_nrem = bouts_q.groupby(["rat", "date"])["hour1_min"].sum()
good_days = day_nrem[day_nrem >= MIN_NREM_MIN].reset_index()[["rat", "date"]]

print(
    f"rat-days: {day_nrem.shape[0]} total -> {len(good_days)} with >= {MIN_NREM_MIN} min NREM "
    f"in the first hour ({day_nrem.shape[0] - len(good_days)} dropped)"
)

rip_q = rip.merge(good_days, on=["rat", "date"])
spn_q = spn.merge(good_days, on=["rat", "date"])

SEQ_WINDOWS = [("seq", SEQ_WINDOW_MS[0], SEQ_WINDOW_MS[1])]


def ratio_per_rat_seq(rip_df, spn_df, r_area, s_area, split_col):
    """One sequential-window ratio per rat per level of split_col, for one
    pair. No pair-count cutoff -- every rat/level with at least one pair is
    kept, and the raw pair count ('pairs') is returned alongside the ratio
    so thin cells stay visible rather than being silently dropped."""
    r = rip_df[rip_df["area"] == r_area]
    s_all = spn_df[spn_df["area"] == s_area]
    rows = []
    for (rat, level), rg in r.groupby(["rat", split_col], observed=True):
        sg = s_all[(s_all["rat"] == rat) & (s_all[split_col] == level)]
        if sg.empty:
            continue
        lg = lh.collect_lags_cols(
            rg, sg, "ripple_end_index", "spindle_start_index", max_lag_ms=MAX_LAG_MS
        )
        if len(lg) == 0:
            continue
        res = lh.summarise(
            lg, EDGES, baseline_ms=BASELINE_MS, windows=SEQ_WINDOWS, verbose=False
        )
        rows.append(
            {
                "rat": rat,
                split_col: level,
                "pair": f"{r_area}_{s_area}",
                "r_area": r_area,
                "s_area": s_area,
                "ratio": res["windows"]["seq"]["ratio"],
                "peak_ms": res["peak_ms"],
                "pairs": len(lg),
            }
        )
    return pd.DataFrame(rows)


def compare_seq(table, split_col, levels, ref, pairs=None):
    """Paired comparison of each level against `ref`, per pair. Prints the
    raw pair count for every rat/level alongside the ratio."""
    pairs = pairs or PAIRS
    for r_area, s_area in pairs:
        pair_label = f"{r_area} ripple - {s_area} spindle"
        w = (
            table[table["pair"] == f"{r_area}_{s_area}"]
            .pivot(index="rat", columns=split_col, values="ratio")
            .dropna()
        )
        n = table[table["pair"] == f"{r_area}_{s_area}"].pivot(
            index="rat", columns=split_col, values="pairs"
        )
        if len(w) < 3:
            print(f"{pair_label}: only {len(w)} rats -- skipping")
            continue
        print(f"\n{pair_label}  (n={len(w)} rats)")
        print("  pairs per rat:")
        print(n.loc[w.index].to_string())
        print("  means: " + " | ".join(f"{l} {w[l].mean():.3f}" for l in levels))
        for l in [x for x in levels if x != ref]:
            d = w[l] - w[ref]
            print(
                f"  {l:>7} vs {ref}: {d.mean():+.3f} | "
                f"{int((d > 0).sum())}/{len(w)} rats higher"
            )


hc_tr_seq = pd.concat(
    [
        ratio_per_rat_seq(rip_q, spn_q, r_area, s_area, "group")
        for r_area, s_area in PAIRS
    ],
    ignore_index=True,
)
hc_tr_seq = hc_tr_seq[hc_tr_seq["group"].isin(["HC", "Training"])]

print("rats with data per cell (after NREM-duration filter):")
print(hc_tr_seq.groupby(["pair", "group"])["rat"].nunique().to_string())

In [ ]:
compare_seq(hc_tr_seq, "group", ["HC", "Training"], ref="HC")

---
# Notes

## Not in this notebook yet

- No over-time plots
- No unique-count rule

## For Yixiao

- Same open question as the full dataset: different enough from the overlap measure?
- Less data in the first hour, compare with the full-dataset results.